[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/03_classification/03_classification_solutions.ipynb)

# 03. Classification — 연습 문제 해설

[03_classification.ipynb](03_classification.ipynb) 끝의 **연습 문제 3개**에 대한 정답 코드와
해설입니다. **먼저 직접 시도해본 뒤** 참고하세요.

> **읽는 법** — 셀은 위에서부터 순서대로 실행해야 하고, 실행 결과는 저장되어 있지 않으니
> 직접 실행해야 출력이 나타납니다.


In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q scikit-learn numpy matplotlib koreanize-matplotlib

import numpy as np
import matplotlib.pyplot as plt

try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    import matplotlib.font_manager as fm

    for _name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
        if any(_name == f.name for f in fm.fontManager.ttflist):
            plt.rc("font", family=_name)
            break
plt.rcParams["axes.unicode_minus"] = False

# 본 노트북과 똑같은 데이터
rng = np.random.default_rng(1)
hours = np.linspace(0, 10, 60)
prob_pass = 1 / (1 + np.exp(-(hours - 5)))
passed = (rng.random(len(hours)) < prob_pass).astype(float)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def train_logreg(hs, ys, lr, epochs=500):
    W, b = 0.0, 0.0
    m = len(hs)
    history = []
    eps = 1e-7   # log(0) = -inf 를 피하는 안전장치
    for _ in range(epochs):
        h = sigmoid(W * hs + b)
        history.append(-np.mean(ys * np.log(h + eps) + (1 - ys) * np.log(1 - h + eps)))
        W -= lr * np.mean((h - ys) * hs)
        b -= lr * np.mean(h - ys)
    return W, b, history


print(f"데이터 {len(hours)}명 준비 완료")


## 연습 1. 학습률(`lr`)을 0.01, 1.0으로 바꿔보기


In [ ]:
results = {}
for lr in [0.01, 0.1, 1.0]:
    W, b, hist = train_logreg(hours, passed, lr)
    results[lr] = hist
    print(f"lr={lr:<5} 최종 cost={hist[-1]:.4f}  W={W:.3f}  b={b:.3f}  합격 경계={-b / W:.2f}시간")

plt.figure(figsize=(7, 4))
for lr, hist in results.items():
    plt.plot(hist, label=f"lr={lr}")
plt.xlabel("epoch")
plt.ylabel("교차 엔트로피 cost")
plt.title("학습률에 따른 수렴 속도")
plt.legend()
plt.show()


**해설**

| `lr` | 500 epoch 후 cost | 합격 경계 | 읽는 법 |
|---|---|---|---|
| 0.01 | 0.494 | 2.72시간 | **한참 모자랍니다.** 곡선이 아직 내려가는 중이고 경계도 엉뚱합니다 |
| 0.1 | 0.291 | 4.67시간 | 잘 내려갔지만 아직 완전히 도착하지는 않았습니다 |
| 1.0 | 0.279 | 5.51시간 | **가장 많이 내려갔습니다.** 진동 없이 안정적입니다 |

02번에서는 `lr=0.1`이 발산했는데 여기서는 1.0도 멀쩡한 이유가 뭘까요?
**cost 함수의 모양이 다르기 때문**입니다. MSE는 오차가 커지면 제곱으로 커져서 기울기가 폭발하지만,
시그모이드를 통과한 교차 엔트로피는 기울기가 `(h - y)`라서 **아무리 틀려도 절댓값이 1을 넘지 않습니다.**
그래서 큰 보폭에도 잘 견딥니다.

**"학습률은 0.01이 안전하다" 같은 규칙은 없습니다.** 문제마다 다르고, cost 곡선을 보고 정해야 합니다.


## 연습 2. `StandardScaler`를 빼고 학습하면?

붓꽃 데이터를 스케일링한 경우와 안 한 경우로 각각 학습해 비교합니다.
정확도만 보지 말고 **몇 번 반복해서 수렴했는지(`n_iter_`)** 도 함께 봅니다.


In [ ]:
import warnings

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.exceptions import ConvergenceWarning

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=0, stratify=iris.target
)

# (A) 스케일링 없이 원본 값 그대로
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", ConvergenceWarning)
    clf_raw = LogisticRegression(max_iter=200).fit(X_train, y_train)
    warned = any(issubclass(w.category, ConvergenceWarning) for w in caught)

# (B) 스케일링 적용
scaler = StandardScaler()
clf_scaled = LogisticRegression(max_iter=200).fit(scaler.fit_transform(X_train), y_train)

print(f"(A) 스케일링 없음: 정확도={accuracy_score(y_test, clf_raw.predict(X_test)):.3f}  "
      f"반복 횟수={clf_raw.n_iter_[0]}회  수렴 경고={'있음' if warned else '없음'}")
print(f"(B) 스케일링 적용: 정확도="
      f"{accuracy_score(y_test, clf_scaled.predict(scaler.transform(X_test))):.3f}  "
      f"반복 횟수={clf_scaled.n_iter_[0]}회")


**해설**

- **정확도는 오히려 스케일링을 안 한 쪽이 1.000으로 더 높게 나옵니다.** 당황할 필요 없습니다.
  평가용이 30송이뿐이라 **한 송이 차이가 0.033**입니다. 이 정도는 우연의 범위입니다.
  붓꽃은 특성 4개가 전부 cm 단위라서 애초에 스케일 차이가 크지 않기도 합니다.
- **진짜 차이는 반복 횟수에 있습니다.** 82회 대 14회로, 스케일링을 하면 6분의 1도 안 되는
  반복으로 같은 답에 도착합니다.
  cost 지형이 길쭉한 계곡 모양에서 둥근 그릇 모양으로 바뀌기 때문입니다.
  길쭉한 계곡에서는 경사 하강법이 지그재그로 내려가느라 오래 걸립니다.

**그래서 스케일링은 습관적으로 하는 것이 안전합니다.** 여기서는 손해가 없지만,
'나이(20~70)'와 '연봉(3000만~2억)'을 함께 쓰는 데이터라면 연봉이 학습을 완전히 지배해서
정확도까지 무너집니다. 그런 사례는 `tabular-ml-practice` 02번에서 직접 보게 됩니다.


## 연습 3. 이상치 실험을 로지스틱 회귀로 다시 하면?

1절에서 직선은 "50시간 공부하고 합격한 사람" 한 명 때문에 판정 경계가 5.00 → 5.51시간으로
밀렸습니다. 시그모이드는 정말 안 밀리는지 같은 실험을 해봅니다.


In [ ]:
hours_out = np.append(hours, 50.0)
passed_out = np.append(passed, 1.0)

# (A) 직선 — 1절과 같은 실험
W_lin, b_lin = np.polyfit(hours, passed, 1)
W_lin_out, b_lin_out = np.polyfit(hours_out, passed_out, 1)

# (B) 로지스틱 회귀
W_log, b_log, _ = train_logreg(hours, passed, 0.1)
W_log_out, b_log_out, _ = train_logreg(hours_out, passed_out, 0.1)

print(f"{'':>12} {'이상치 전':>12} {'이상치 후':>12} {'밀린 정도':>10}")
lin_b, lin_a = (0.5 - b_lin) / W_lin, (0.5 - b_lin_out) / W_lin_out
log_b, log_a = -b_log / W_log, -b_log_out / W_log_out
print(f"{'직선':>12} {lin_b:>12.3f} {lin_a:>12.3f} {lin_a - lin_b:>10.3f}")
print(f"{'로지스틱':>12} {log_b:>12.3f} {log_a:>12.3f} {log_a - log_b:>10.3f}")


**해설**

직선은 0.5시간 넘게 밀리는데 **로지스틱 회귀는 소수점 아래까지 거의 그대로**입니다.

이유는 시그모이드 곡선의 모양에 있습니다. 50시간짜리 데이터는 이미 시그모이드의
오른쪽 평평한 구간에 있어서 예측값이 사실상 1입니다. **이미 맞히고 있으니 오차가 0에 가깝고,
오차가 0이면 기울기(`h - y`)도 0이라 학습에 아무 영향을 주지 못합니다.**

반면 직선은 x=50에서 예측값이 8이든 10이든 계속 커지기 때문에, 오차도 계속 커지고
그 오차를 줄이려고 기울기를 낮춥니다.

**"틀린 만큼만 배우고, 이미 맞힌 것에는 끌려다니지 않는다."** 시그모이드가 분류에 적합한
이유가 여기에 있습니다.
